In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time # Added for tracking training speed

# --- Week 9 Dataset ---
dataset_week9 = [
    "machine learning models learn patterns from data.",
    "sequence models process data step by step.",
    "recurrent neural networks are designed for sequential tasks.",
    "rnn models maintain hidden states across time steps.",
    "long short term memory networks solve long dependency problems.",
    "lstm uses gates to control information flow.",
    "gru models simplify the lstm architecture.",
    "sequence prediction is useful in many applications.",
    "language modeling predicts the next word in a sentence.",
    "speech recognition processes audio sequences.",
    "time series forecasting predicts future values.",
    "music generation creates new melodies.",
    "generative models learn probability distributions.",
    "they generate new samples similar to training data.",
    "sequence generation is widely used in artificial intelligence.",
    "deep learning improves sequence modeling performance."
]

# --- Week 10 Dataset ---
dataset_week10 = [
    "artificial intelligence systems learn patterns from data.",
    "sequence models process information step by step.",
    "recurrent neural networks are useful for sequence prediction.",
    "lstm networks handle long term dependencies.",
    "deep learning models improve sequence learning.",
    "generative models create new samples from learned patterns.",
    "language models predict the next word in a sentence.",
    "sequence generation is used in chatbots and assistants.",
    "machine learning helps computers learn automatically.",
    "training data improves model accuracy.",
    "neural networks simulate human brain structures.",
    "optimization algorithms improve learning efficiency.",
    "technology is transforming modern education.",
    "online learning platforms use artificial intelligence.",
    "students benefit from intelligent tutoring systems.",
    "automation improves productivity and decision making."
]

# CHANGE THIS to switch between Week 9 and Week 10
corpus = dataset_week9

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

# Create Input-Output Sequence Pairs
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences for uniform length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split into Predictors (X) and Label (y)
X, labels = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(labels, num_classes=total_words)

print(f"Total Words: {total_words}")
print(f"Max Sequence Length: {max_sequence_len}")

# Universal Generation Function
def generate_text(seed_text, next_words, model, max_seq_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

Total Words: 87
Max Sequence Length: 9


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

print("--- Building RNN Model ---")
model_rnn = Sequential([
    Input(shape=(max_sequence_len-1,)),
    Embedding(input_dim=total_words, output_dim=64),
    SimpleRNN(100),
    Dense(total_words, activation='softmax')
])

model_rnn.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_rnn.summary() # MEMORY USAGE: Check "Total params" here

print("\nTraining RNN...")
start_time = time.time()
model_rnn.fit(X, y, epochs=100, verbose=0) # verbose=0 to keep output clean, change to 1 to see progress
end_time = time.time()

print(f"-> RNN Training Time: {end_time - start_time:.2f} seconds") # TRAINING SPEED

print("\n--- RNN Generated Text (Context Retention) ---")
print(generate_text("sequence models", 6, model_rnn, max_sequence_len))

--- Building RNN Model ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 8, 64)          │         5,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 100)            │        16,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 87)             │         8,787 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,855 (120.53 KB)

 Trainable params: 30,855 (120.53 KB)

 Non-trainable params: 0 (0.00 B)


Training RNN...
-> RNN Training Time: 19.19 seconds

--- RNN Generated Text (Context Retention) ---
sequence models process data step by step intelligence


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

print("--- Building LSTM Model ---")
model_lstm = Sequential([
    Input(shape=(max_sequence_len-1,)),
    Embedding(input_dim=total_words, output_dim=64),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary() # MEMORY USAGE: Check "Total params", it will be higher than RNN

print("\nTraining LSTM...")
start_time = time.time()
model_lstm.fit(X, y, epochs=100, verbose=0)
end_time = time.time()

print(f"-> LSTM Training Time: {end_time - start_time:.2f} seconds") # TRAINING SPEED

print("\n--- LSTM Generated Text (Context Retention) ---")
print(generate_text("sequence models", 6, model_lstm, max_sequence_len))

--- Building LSTM Model ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 8, 64)          │         5,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100)            │        66,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 87)             │         8,787 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,355 (313.89 KB)

 Trainable params: 80,355 (313.89 KB)

 Non-trainable params: 0 (0.00 B)


Training LSTM...
-> LSTM Training Time: 13.50 seconds

--- LSTM Generated Text (Context Retention) ---
sequence models process data step by step step


In [4]:
from tensorflow.keras import layers, Model
import tensorflow as tf

print("--- Building Transformer Model ---")

# Positional Encoding
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# Architecture
embed_dim = 64
num_heads = 2
ff_dim = 64

inputs = layers.Input(shape=(max_sequence_len-1,))
embedding_layer = TokenAndPositionEmbedding(max_sequence_len-1, total_words, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(total_words, activation="softmax")(x)

model_transformer = Model(inputs=inputs, outputs=outputs)
model_transformer.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_transformer.summary() # MEMORY USAGE: Check "Total params"

print("\nTraining Transformer...")
start_time = time.time()
model_transformer.fit(X, y, epochs=100, verbose=0)
end_time = time.time()

print(f"-> Transformer Training Time: {end_time - start_time:.2f} seconds") # TRAINING SPEED

print("\n--- Transformer Generated Text (Context Retention) ---")
print(generate_text("sequence models", 6, model_transformer, max_sequence_len))

--- Building Transformer Model ---


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 8, 64)          │         6,080 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 8, 64)          │        41,792 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 87)             │         5,655 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,527 (209.09 KB)

 Trainable params: 53,527 (209.09 KB)

 Non-trainable params: 0 (0.00 B)


Training Transformer...
-> Transformer Training Time: 16.83 seconds

--- Transformer Generated Text (Context Retention) ---
sequence models process data step by step by
